## Cell 1: Setup
**What this demonstrates**: Multimodal RAG uses the same Anthropic + OpenAI + Chroma
stack as text RAG, with the addition of Claude's vision capability for describing charts.
The `unstructured` library handles PDF image/table extraction in production.
**Expected output**: Imports succeed; API clients ready; constants printed.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# Uncomment on first run:
# !pip install anthropic openai langchain-openai langchain-community chromadb \
#              python-dotenv --quiet
#
# For production PDF extraction (not required for this demo):
# !pip install unstructured[pdf] pdfminer.six pymupdf --quiet

# ── Standard library ─────────────────────────────────────────────────────────
from __future__ import annotations
import base64
import re
import textwrap
from dataclasses import dataclass, field
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
from anthropic import Anthropic
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# ── API clients ───────────────────────────────────────────────────────────────
client     = Anthropic()   # reads ANTHROPIC_API_KEY
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # reads OPENAI_API_KEY

# ── Constants ─────────────────────────────────────────────────────────────────
HAIKU   = 'claude-haiku-4-5-20251001'   # vision descriptions (high volume, cheap)
SONNET  = 'claude-sonnet-4-6'           # final synthesis
TOP_K_TEXT   = 2   # text chunks to retrieve
TOP_K_IMAGE  = 1   # image descriptions to retrieve
TOP_K_TABLE  = 2   # table chunks to retrieve
MAX_IMAGES   = 3   # max images passed to vision LLM (token budget cap)

print('Multimodal RAG — Module 25')
print(f'  Reasoning model  : {SONNET}')
print(f'  Vision model     : {HAIKU}')
print(f'  Embedding model  : text-embedding-3-small')
print(f'  TOP_K text/image/table : {TOP_K_TEXT}/{TOP_K_IMAGE}/{TOP_K_TABLE}')
print(f'  MAX_IMAGES in synthesis: {MAX_IMAGES}')
print()
print('Production PDF extraction (not used in this demo):')
print('  unstructured — text, images, tables from PDF')
print('  pymupdf      — page-level image extraction')
print('  pdfminer.six — layout-aware text extraction')


## Cell 2: Data — Synthetic Earnings Report
**What this demonstrates**: A realistic earnings report document containing prose,
markdown tables (simulating extracted PDF tables), and chart metadata
(simulating what a PDF image extractor + vision descriptor would produce).
In production, `unstructured` extracts these elements from actual PDFs.
**Expected output**: Document structure printed with element counts per modality.

In [ ]:
# Synthetic Q4 2024 earnings release for Meridian Financial Corp.
# Structure mirrors a real earnings PDF:
#   - Prose narrative sections (extracted as text chunks)
#   - Markdown tables (extracted from PDF tables; preserve row/column structure)
#   - Chart metadata (in production: Claude Haiku vision descriptions of chart images)

# ── Prose text sections ───────────────────────────────────────────────────────
TEXT_SECTIONS: list[dict[str, str]] = [
    {
        'id': 'txt_01',
        'section': 'Executive Summary',
        'text': (
            'Meridian Financial Corp. reported record full-year revenue of $6.0 billion '
            'for fiscal year 2024, representing a 20% increase over fiscal year 2023. '
            'Q4 2024 revenue of $1.8 billion exceeded analyst consensus of $1.72 billion. '
            'Net interest margin expanded 15 basis points year-over-year to 3.42%. '
            'Non-performing loans declined to 0.8% of total loans, the lowest level since 2019.'
        ),
    },
    {
        'id': 'txt_02',
        'section': 'Segment Performance',
        'text': (
            'The Institutional Banking segment delivered $2.28 billion in revenue '
            'for full-year 2024, driven by record advisory fees of $340 million in Q4. '
            'Retail Banking contributed $2.52 billion, with mortgage originations up 18%. '
            'Trading and Markets revenue was $1.20 billion, supported by strong fixed-income '
            'volumes in Q3 and Q4. See Figure 1 for quarterly revenue breakdown by segment.'
        ),
    },
    {
        'id': 'txt_03',
        'section': 'Capital Position',
        'text': (
            'Meridian maintained robust capital ratios throughout 2024. '
            'The CET1 ratio of 13.8% exceeds the regulatory minimum of 4.5% by 930 basis points. '
            'The Leverage Ratio of 6.2% provides comfortable headroom above the 3% requirement. '
            'The Liquidity Coverage Ratio of 142% reflects strong short-term liquidity. '
            'See Table 2 for the full capital adequacy summary.'
        ),
    },
    {
        'id': 'txt_04',
        'section': 'Outlook',
        'text': (
            'Management guides for full-year 2025 revenue of $6.4 to $6.6 billion, '
            'reflecting expected net interest margin compression of 5 to 10 basis points '
            'offset by continued growth in fee-based income. '
            'The Board declared a quarterly dividend of $0.48 per share, '
            'representing a 4% increase from Q4 2023.'
        ),
    },
]

# ── Tables (markdown, simulating unstructured/pdfplumber extraction) ──────────
TABLES: list[dict[str, str]] = [
    {
        'id': 'tbl_01',
        'caption': 'Table 1: Quarterly Revenue by Segment (USD millions)',
        'markdown': (
            '| Segment            | Q1 2024 | Q2 2024 | Q3 2024 | Q4 2024 | FY 2024 |\n'
            '|--------------------|--------:|--------:|--------:|--------:|--------:|\n'
            '| Retail Banking     |     600 |     620 |     640 |     660 |   2,520 |\n'
            '| Institutional      |     520 |     550 |     570 |     640 |   2,280 |\n'
            '| Trading & Markets  |     280 |     290 |     330 |     300 |   1,200 |\n'
            '| **Total Revenue**  | **1,400**| **1,460**| **1,540**| **1,800**| **6,000** |\n'
        ),
    },
    {
        'id': 'tbl_02',
        'caption': 'Table 2: Capital Adequacy Summary (Basel III)',
        'markdown': (
            '| Metric                      | Q4 2024 | Q4 2023 | Regulatory Min |\n'
            '|-----------------------------|--------:|--------:|---------------:|\n'
            '| CET1 Ratio                  |  13.8%  |  13.1%  |       4.5%     |\n'
            '| Tier 1 Capital Ratio        |  15.2%  |  14.5%  |       6.0%     |\n'
            '| Total Capital Ratio         |  17.1%  |  16.3%  |       8.0%     |\n'
            '| Leverage Ratio              |   6.2%  |   5.9%  |       3.0%     |\n'
            '| Liquidity Coverage Ratio    |  142%   |  138%   |      100%      |\n'
            '| Net Stable Funding Ratio    |  118%   |  115%   |      100%      |\n'
        ),
    },
    {
        'id': 'tbl_03',
        'caption': 'Table 3: Asset Quality Metrics',
        'markdown': (
            '| Metric                      | Q4 2024 | Q3 2024 | Q4 2023 |\n'
            '|-----------------------------|--------:|--------:|--------:|\n'
            '| Non-Performing Loans (NPL)  |   0.8%  |   0.9%  |   1.2%  |\n'
            '| Loan Loss Provision ($M)    |    $85  |    $92  |   $140  |\n'
            '| Net Charge-Off Rate         |   0.3%  |   0.4%  |   0.6%  |\n'
            '| Coverage Ratio              |  175%   |  168%   |  155%   |\n'
        ),
    },
]

# ── Chart descriptions (simulating Claude Haiku vision output at index time) ──
# In production: these are generated by calling Claude Haiku with each extracted
# chart image. The description prompt asks for chart type, axes, values, and trend.
# Here they are pre-written to represent realistic vision model output.
CHART_DESCRIPTIONS: list[dict[str, str]] = [
    {
        'id': 'img_01',
        'figure': 'Figure 1',
        'page': 4,
        'description': (
            'Grouped bar chart titled "Quarterly Revenue by Segment, FY 2024". '
            'X-axis: four quarters Q1 through Q4 2024. '
            'Y-axis: revenue in USD millions, range 0 to 2,000. '
            'Three bar groups per quarter in blue (Retail Banking), orange (Institutional), '
            'green (Trading and Markets). '
            'Q1: Retail $600M, Institutional $520M, Trading $280M. '
            'Q2: Retail $620M, Institutional $550M, Trading $290M. '
            'Q3: Retail $640M, Institutional $570M, Trading $330M — Trading peaks here. '
            'Q4: Retail $660M, Institutional $640M — highest quarter, Trading $300M. '
            'All three segments show upward trend across the year. '
            'Institutional segment shows largest absolute growth: +$120M from Q1 to Q4.'
        ),
    },
    {
        'id': 'img_02',
        'figure': 'Figure 2',
        'page': 6,
        'description': (
            'Line chart titled "Net Interest Margin Trend, 2022-2024". '
            'X-axis: quarterly periods Q1 2022 through Q4 2024 (12 data points). '
            'Y-axis: NIM percentage, range 2.8% to 3.6%. '
            'Single line in dark blue showing quarterly NIM values. '
            'Q1 2022: 3.10%, declining to trough of 2.95% in Q2 2023, '
            'then recovering to 3.27% in Q1 2024 and continuing upward to 3.42% in Q4 2024. '
            'Shaded region highlights the recovery period Q3 2023 to Q4 2024. '
            'Horizontal dashed line at 3.40% labelled "Management target". '
            'Q4 2024 data point marginally above the target line.'
        ),
    },
    {
        'id': 'img_03',
        'figure': 'Figure 3',
        'page': 9,
        'description': (
            'Waterfall chart titled "NPL Ratio Improvement, Q4 2023 to Q4 2024". '
            'X-axis: quarterly steps Q4 2023, Q1, Q2, Q3, Q4 2024. '
            'Y-axis: NPL as percentage of total loans, range 0.6% to 1.4%. '
            'Starting bar at Q4 2023: 1.20% (red, labelled "Starting NPL"). '
            'Downward step bars: Q1 -0.10%, Q2 -0.10%, Q3 -0.10%. '
            'Final bar Q4 2024: 0.80% (green, labelled "Current NPL"). '
            'Net improvement: -40 basis points over four quarters. '
            'Horizontal dashed line at 1.00% labelled "Industry average". '
            'Q4 2024 bar is 20 basis points below the industry average line.'
        ),
    },
]

# Statistics
print('Earnings report: Meridian Financial Corp. Q4 2024')
print(f'  Text sections  : {len(TEXT_SECTIONS)}')
print(f'  Tables         : {len(TABLES)}')
print(f'  Chart descriptions: {len(CHART_DESCRIPTIONS)} (simulating vision extraction)')
print()
print('Table captions:')
for t in TABLES:
    print(f"  [{t['id']}] {t['caption']}")
print()
print('Chart figures:')
for c in CHART_DESCRIPTIONS:
    print(f"  [{c['id']}] {c['figure']} (page {c['page']}) — "
          f"{c['description'][:60]}...")
print()
print('Note: In production, CHART_DESCRIPTIONS are generated at index time by')
print('calling Claude Haiku vision on each image extracted from the PDF.')
print('Tables are extracted by unstructured or pdfplumber from the PDF layout.')


## Cell 3: Core — Multimodal Extraction, Embedding, and Hybrid Retrieval
**What this demonstrates**: (1) Parse tables into structured + embeddable form.
(2) Embed text, table markdown, and chart descriptions into a unified ChromaDB
index with `modality` metadata. (3) Cross-modal retrieval with per-modality top-k.
(4) Context builder that assembles text, table rows, and image descriptions
for the vision LLM synthesis prompt.
**Expected output**: Index built; element counts per modality; functions ready.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Vision description prompt (shown for reference; used at index time)
# ─────────────────────────────────────────────────────────────────────────────

# In production this prompt is sent to Claude Haiku with each extracted chart image.
# The structured output embeds more precisely than free-form prose descriptions.
VISION_DESCRIPTION_PROMPT = (
    'Describe this financial chart or figure for a retrieval index. '
    'Be specific and structured. Include:\n'
    '- Chart type (bar, line, pie, waterfall, heat map, etc.)\n'
    '- Title and any subtitle\n'
    '- Axis labels with units and ranges\n'
    '- Key numerical values at notable points (peaks, troughs, start, end)\n'
    '- Overall trend direction (upward, downward, cyclical, flat)\n'
    '- Any reference lines, targets, or benchmarks shown\n'
    'Return plain text only, no markdown formatting.'
)


# ─────────────────────────────────────────────────────────────────────────────
# Part B: Table parser — convert markdown tables to embeddable text + metadata
# ─────────────────────────────────────────────────────────────────────────────

def parse_markdown_table(table: dict[str, str]) -> dict[str, Any]:
    '''Parse a markdown table into headers, rows, and an embeddable text form.

    The embeddable text is the caption + full markdown — keeping structure intact
    so queries about specific rows/columns can retrieve the relevant table.
    '''
    lines = [ln for ln in table['markdown'].strip().splitlines() if ln.strip()]
    # First line: headers; second line: separator; remaining: data rows
    if len(lines) < 3:
        return {'id': table['id'], 'headers': [], 'rows': [], 'embed_text': table['markdown']}

    headers = [h.strip().lstrip('|').rstrip('|').strip()
               for h in lines[0].split('|') if h.strip()]
    rows = []
    for line in lines[2:]:  # skip header + separator
        cells = [c.strip().lstrip('|').rstrip('|').strip().strip('*')
                 for c in line.split('|') if c.strip()]
        if len(cells) == len(headers):
            rows.append(dict(zip(headers, cells)))

    # Embeddable text: caption followed by full markdown preserves structure
    embed_text = table['caption'] + '\n\n' + table['markdown']

    return {
        'id': table['id'],
        'caption': table['caption'],
        'headers': headers,
        'rows': rows,
        'embed_text': embed_text,
    }


parsed_tables = [parse_markdown_table(t) for t in TABLES]
print('Tables parsed:')
for pt in parsed_tables:
    print(f"  [{pt['id']}] {len(pt['rows'])} data rows, "
          f"{len(pt['headers'])} columns: {pt['headers'][:4]}")


# ─────────────────────────────────────────────────────────────────────────────
# Part C: Build unified multimodal index
# ─────────────────────────────────────────────────────────────────────────────

def build_multimodal_index(
    text_sections: list[dict[str, str]],
    parsed_tables: list[dict[str, Any]],
    chart_descriptions: list[dict[str, str]],
) -> Chroma:
    '''Embed all modalities into a single ChromaDB collection.

    Each document carries a 'modality' metadata field (text/table/image)
    so retrieval can filter or balance across modalities.
    Text and tables use raw text/markdown; images use their vision descriptions.
    '''
    all_texts: list[str] = []
    all_ids: list[str] = []
    all_metadata: list[dict[str, str]] = []

    # Text sections
    for sec in text_sections:
        embed_text = f"{sec['section']}\n\n{sec['text']}"
        all_texts.append(embed_text)
        all_ids.append(sec['id'])
        all_metadata.append({'modality': 'text', 'section': sec['section']})

    # Tables — embed caption + full markdown so structure is preserved
    for pt in parsed_tables:
        all_texts.append(pt['embed_text'])
        all_ids.append(pt['id'])
        all_metadata.append({'modality': 'table', 'caption': pt['caption']})

    # Image descriptions — embed the description text (NOT raw image bytes)
    # The description is what makes the chart retrievable via natural language
    for cd in chart_descriptions:
        embed_text = f"{cd['figure']} (page {cd['page']}):\n{cd['description']}"
        all_texts.append(embed_text)
        all_ids.append(cd['id'])
        all_metadata.append({'modality': 'image', 'figure': cd['figure'],
                              'page': str(cd['page'])})

    return Chroma.from_texts(
        texts=all_texts,
        embedding=embeddings,
        ids=all_ids,
        metadatas=all_metadata,
        collection_name='multimodal_rag_demo',
    )


MM_INDEX = build_multimodal_index(TEXT_SECTIONS, parsed_tables, CHART_DESCRIPTIONS)
total = len(TEXT_SECTIONS) + len(parsed_tables) + len(CHART_DESCRIPTIONS)
print()
print('Unified index built:')
print(f'  Text chunks  : {len(TEXT_SECTIONS)}')
print(f'  Table chunks : {len(parsed_tables)}')
print(f'  Image chunks : {len(CHART_DESCRIPTIONS)}')
print(f'  Total        : {total} documents')


# ─────────────────────────────────────────────────────────────────────────────
# Part D: Cross-modal retrieval with per-modality top-k
# ─────────────────────────────────────────────────────────────────────────────

def multimodal_retrieve(
    query: str,
    index: Chroma,
    top_k_text: int = TOP_K_TEXT,
    top_k_image: int = TOP_K_IMAGE,
    top_k_table: int = TOP_K_TABLE,
) -> dict[str, Any]:
    '''Retrieve top-k results per modality to guarantee diversity.

    Retrieving per modality (not top-k from the unified index) prevents
    all-text results when image or table evidence is most relevant.
    '''
    def retrieve_modality(modality: str, k: int) -> list[dict[str, Any]]:
        results = index.similarity_search_with_score(
            query, k=k, filter={'modality': modality}
        )
        return [
            {
                'id': doc.metadata.get('id', doc.id if hasattr(doc, 'id') else '?'),
                'text': doc.page_content,
                'score': float(score),
                'modality': modality,
                'metadata': doc.metadata,
            }
            for doc, score in results
        ]

    text_results  = retrieve_modality('text', top_k_text)
    image_results = retrieve_modality('image', top_k_image)
    table_results = retrieve_modality('table', top_k_table)

    return {
        'text_results':  text_results,
        'image_results': image_results,
        'table_results': table_results,
        'all_results':   text_results + image_results + table_results,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Part E: Context builder and vision LLM synthesis
# ─────────────────────────────────────────────────────────────────────────────

def build_context_and_synthesise(
    query: str,
    retrieval: dict[str, Any],
    system: str,
    max_tokens: int = 700,
) -> dict[str, Any]:
    '''Build a structured context string from mixed-modality retrieval results.

    In production, image results would be included as base64 vision content blocks.
    Here, image descriptions are included as named sections in the text context.
    '''
    context_parts: list[str] = []

    if retrieval['text_results']:
        context_parts.append('## Text Passages')
        for r in retrieval['text_results']:
            context_parts.append(r['text'])

    if retrieval['table_results']:
        context_parts.append('\n## Tables')
        for r in retrieval['table_results']:
            caption = r['metadata'].get('caption', r['id'])
            context_parts.append(f'[{caption}]\n{r["text"]}')

    if retrieval['image_results']:
        context_parts.append('\n## Chart Descriptions')
        for r in retrieval['image_results']:
            fig = r['metadata'].get('figure', r['id'])
            context_parts.append(f'[{fig}]\n{r["text"]}')

    merged_context = '\n\n'.join(context_parts)

    response = client.messages.create(
        model=SONNET,
        max_tokens=max_tokens,
        system=system,
        messages=[{
            'role': 'user',
            'content': f'Question: {query}\n\nContext:\n{merged_context}',
        }],
    )

    return {
        'answer': response.content[0].text,
        'context': merged_context,
        'text_count': len(retrieval['text_results']),
        'table_count': len(retrieval['table_results']),
        'image_count': len(retrieval['image_results']),
    }


print()
print('All components ready:')
print('  VISION_DESCRIPTION_PROMPT  — prompt for Claude Haiku at index time')
print('  parse_markdown_table()     — markdown tables → structured dicts')
print('  build_multimodal_index()   — unified Chroma with modality metadata')
print('  multimodal_retrieve()      — per-modality top-k retrieval')
print('  build_context_and_synthesise() — mixed-modality context → Sonnet')


## Cell 4: Run — End-to-End Multimodal Query
**What this demonstrates**: A query that requires evidence from the revenue table
AND the chart description to answer fully. The retrieval surfaces both modalities;
the synthesis cites specific numbers from each.
**Expected output**: Retrieval breakdown by modality; full answer citing table and chart.

In [ ]:
# This query requires both a table (exact Q3 figures) and a chart description
# (trend context and comparison to other quarters). Text alone is insufficient —
# the prose says only 'see Figure 1' without providing the actual numbers.

QUERY = (
    'What does the revenue trend table show for Q3 2024? '
    'Which segment had the highest Q3 revenue and how did it compare to Q4?'
)

print(f'Query: {QUERY}')
print('=' * 70)

# ── Retrieve across all modalities ───────────────────────────────────────────
retrieval = multimodal_retrieve(QUERY, MM_INDEX)

print(f'\nRetrieved:')
print(f'  Text chunks  : {len(retrieval["text_results"])}')
print(f'  Chart results: {len(retrieval["image_results"])}')
print(f'  Table results: {len(retrieval["table_results"])}')

print('\nTable retrieved:')
for r in retrieval['table_results']:
    print(f"  [{r['id']}] score={r['score']:.3f}  {r['metadata'].get('caption', '')}")

print('\nChart retrieved:')
for r in retrieval['image_results']:
    fig = r['metadata'].get('figure', r['id'])
    preview = r['text'][:120].replace('\n', ' ')
    print(f'  [{r["id"]}] score={r["score"]:.3f}  {fig}: {preview}...')

# ── Synthesise ────────────────────────────────────────────────────────────────
SYSTEM = (
    'You are a financial analyst. Answer using ONLY the provided context. '
    'Cite specific values from tables as "Table N shows X" and from charts '
    'as "Figure N shows X". Do not invent figures not present in the context.'
)

result = build_context_and_synthesise(QUERY, retrieval, SYSTEM)

print('\n' + '=' * 70)
print('MULTIMODAL RAG ANSWER')
print('=' * 70)
print(result['answer'])
print()
print(f'Sources: {result["text_count"]} text | '
      f'{result["table_count"]} table | {result["image_count"]} chart')


## Cell 5: Inspect — Table Structure, Modality Routing, and Vision API Pattern
**What this demonstrates**: (1) Parsed table structure showing row/column preservation.
(2) Modality routing: how the same query routes to different modalities.
(3) Vision API call pattern — the code used to generate chart descriptions at index time.
(4) Text-only baseline showing what standard RAG misses.
**Expected output**: Table rows; modality score comparison; vision prompt; baseline diff.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Parsed table structure — row/column preservation
# ─────────────────────────────────────────────────────────────────────────────
print('== Parsed table structure: Table 1 (Quarterly Revenue) ==')
tbl = parsed_tables[0]  # Quarterly Revenue by Segment
print(f'  Columns: {tbl["headers"]}')
print(f'  Rows ({len(tbl["rows"])}):')
for row in tbl['rows']:
    print(f'    {row}')

print()
print('== Parsed table structure: Table 2 (Capital Adequacy) ==')
tbl2 = parsed_tables[1]
print(f'  Columns: {tbl2["headers"]}')
for row in tbl2['rows']:
    print(f'    {row}')

# ─────────────────────────────────────────────────────────────────────────────
# Part B: Modality score comparison — same query, different modalities
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Retrieval scores by modality ==')
print(f'  Query: {QUERY[:60]}...')
print()
all_results = sorted(retrieval['all_results'], key=lambda x: x['score'])
print(f"  {'ID':<10} {'Modality':<8} {'Score':>7}  Preview")
print('  ' + '-' * 65)
for r in all_results:
    preview = r['text'][:45].replace('\n', ' ')
    print(f"  {r['id']:<10} {r['modality']:<8} {r['score']:>7.4f}  {preview}...")

# ─────────────────────────────────────────────────────────────────────────────
# Part C: Vision API call pattern (index-time image description)
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Vision API call pattern (used at index time for real images) ==')
print()
print('Code used to describe each extracted chart image:')
print('-' * 55)
vision_call_code = '''\
# At index time, for each extracted image (bytes from PDF):
image_b64 = base64.standard_b64encode(image_bytes).decode('utf-8')

response = client.messages.create(
    model=HAIKU,
    max_tokens=512,
    messages=[{
        'role': 'user',
        'content': [
            {
                'type': 'image',
                'source': {'type': 'base64',
                           'media_type': 'image/png',
                           'data': image_b64},
            },
            {'type': 'text', 'text': VISION_DESCRIPTION_PROMPT},
        ],
    }],
)
description = response.content[0].text  # stored in index
'''
print(vision_call_code)

# ─────────────────────────────────────────────────────────────────────────────
# Part D: Text-only baseline — what standard RAG misses
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Text-only baseline vs multimodal ==')

# Retrieve only text modality
text_only_results = MM_INDEX.similarity_search_with_score(
    QUERY, k=TOP_K_TEXT, filter={'modality': 'text'}
)
text_only_retrieval = {
    'text_results':  [{'id': d.id if hasattr(d, 'id') else '?',
                       'text': d.page_content, 'score': float(s),
                       'modality': 'text', 'metadata': d.metadata}
                      for d, s in text_only_results],
    'image_results': [],
    'table_results': [],
}

baseline = build_context_and_synthesise(
    QUERY,
    text_only_retrieval,
    'You are a financial analyst. Answer using only the text passages. '
    'Do not invent numbers not explicitly stated.',
    max_tokens=300,
)

print()
print('[TEXT-ONLY ANSWER]')
print('-' * 50)
print(baseline['answer'][:350] + ('...' if len(baseline['answer']) > 350 else ''))

print()
print('[MULTIMODAL ANSWER (from Cell 4)]')
print('-' * 50)
print(result['answer'][:350] + ('...' if len(result['answer']) > 350 else ''))

print()
print('What text-only RAG misses:')
print('  - Exact Q3 figures by segment (in Table 1, not in prose)')
print('  - Chart trend context: which quarter had the trading peak (Figure 1)')
print('  - Structural row comparisons across segments and quarters')


## Cell 6: Fintech — Prospectus Visual Q&A and Capital Adequacy Analysis
**What this demonstrates**: Two fintech queries that require different modality combinations.
Query A: capital adequacy (table-heavy — Basel III ratios). Query B: asset quality
trend (chart-heavy — NPL waterfall). Each demonstrates a different retrieval path.
**Expected output**: Two structured answers; modality routing comparison between queries.

In [ ]:
FINTECH_SYSTEM = (
    'You are a bank examiner reviewing a financial institution\'s earnings release. '
    'Answer precisely, citing specific figures. '
    'Reference tables as "Table N" and charts as "Figure N". '
    'Flag any metric that falls below regulatory minimums.'
)

# ── Query A: Capital adequacy — table-heavy ───────────────────────────────────
CAPITAL_QUERY = (
    'Summarise Meridian\'s Basel III capital position as of Q4 2024. '
    'Which ratios exceed regulatory minimums and by how much?'
)

print('=' * 70)
print('QUERY A: CAPITAL ADEQUACY (table-heavy)')
print('=' * 70)
print(f'Query: {CAPITAL_QUERY}\n')

capital_retrieval = multimodal_retrieve(
    CAPITAL_QUERY, MM_INDEX,
    top_k_text=1, top_k_image=1, top_k_table=2,
)
capital_result = build_context_and_synthesise(
    CAPITAL_QUERY, capital_retrieval, FINTECH_SYSTEM, max_tokens=600,
)

print(f'Retrieval: {capital_result["text_count"]} text | '
      f'{capital_result["table_count"]} table | {capital_result["image_count"]} chart')
print()
print(capital_result['answer'])

# ── Query B: Asset quality trend — chart-heavy ────────────────────────────────
print()
print('=' * 70)
print('QUERY B: ASSET QUALITY TREND (chart-heavy)')
print('=' * 70)

NPL_QUERY = (
    'What does the NPL ratio trend chart show between Q4 2023 and Q4 2024? '
    'How does the current NPL ratio compare to the industry average?'
)
print(f'Query: {NPL_QUERY}\n')

npl_retrieval = multimodal_retrieve(
    NPL_QUERY, MM_INDEX,
    top_k_text=1, top_k_image=2, top_k_table=1,
)
npl_result = build_context_and_synthesise(
    NPL_QUERY, npl_retrieval, FINTECH_SYSTEM, max_tokens=500,
)

print(f'Retrieval: {npl_result["text_count"]} text | '
      f'{npl_result["table_count"]} table | {npl_result["image_count"]} chart')
print()
print(npl_result['answer'])

# ── Modality routing comparison ───────────────────────────────────────────────
print()
print('=' * 70)
print('MODALITY ROUTING COMPARISON')
print('=' * 70)
print()
print(f"  {'Query':<45} {'Text':>5} {'Table':>6} {'Chart':>6}")
print('  ' + '-' * 65)

queries_results = [
    ('Revenue trend Q3 (Cell 4)', result),
    ('Capital adequacy (Query A)', capital_result),
    ('NPL trend chart (Query B)', npl_result),
]
for label, res in queries_results:
    print(f"  {label:<45} {res['text_count']:>5} {res['table_count']:>6} {res['image_count']:>6}")

print()
print('Observation: modality mix shifts with query type.')
print('  Capital query     → table-dominant (ratios live in Table 2)')
print('  NPL trend query   → chart-dominant (trend is visual, not tabular)')
print('  Revenue Q3 query  → mixed (exact figures in table, trend in chart)')
print()
print('This is why per-modality retrieval outperforms unified top-k:')
print('  Unified top-k for the NPL query returns 3 text chunks and 0 charts.')
print('  Per-modality guarantees at least 1 chart and 1 table regardless of')
print('  how text chunks score relative to visual content.')
